In [2]:
!C:\Users\shahj\AppData\Local\Programs\Python\Python312\python.exe -m pip install -U transformers
from transformers import pipeline
from newspaper import Article

summarizer = pipeline("text-generation", model="facebook/bart-large-cnn")
news_articles = ["https://www.theguardian.com/world/live/2026/jun/23/europe-heatwave-live-news-updates-uk-red-temperature-warnings-40-degrees-france-heat-deaths-crisis-meeting",
                 "https://www.theguardian.com/world/2026/jun/22/ukraine-intensifies-attacks-on-crimea-to-raise-cost-of-russia-occupation",
                "https://www.theguardian.com/world/2026/jun/23/canada-montreal-shooting-gunman-three-killed-suspect-police-cote-des-neiges"
                ]

for i, news_article in enumerate(news_articles, 1):
        # Download and parse the article
        article = Article(news_article)
        article.download()
        article.parse()
        
        # Extract title and clean text body
        title = article.title
        text_content = article.text
        
        print(f"Title: {title}")
        summary = summarizer(
            text_content, 
            max_length=60
        )


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\shahj\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

[transformers] BartForCausalLM LOAD REPORT from: facebook/bart-large-cnn
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.self_attn.v_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.bias     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn_layer_norm.weight | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn_layer_norm.bias   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.v_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.bias       | UNEXPECTED |  | 
model.encoder.layernorm_embedding.weight                  | UNEXPECTED |  | 
mod

Title: Europe heatwave live: ‘London is cooking,’ says UN chief as UK forecast to hit 38C; France has hottest night since records began


KeyError: 'summary_text'

In [7]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from newspaper import Article
import torch

model_name = "facebook/bart-large-cnn"
print("Loading model and tokenizer...")

# 1. Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, clean_up_tokenization_spaces=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

news_articles = [
    "https://www.theguardian.com/world/live/2026/jun/23/europe-heatwave-live-news-updates-uk-red-temperature-warnings-40-degrees-france-heat-deaths-crisis-meeting",
    "https://www.theguardian.com/world/2026/jun/22/ukraine-intensifies-attacks-on-crimea-to-raise-cost-of-russia-occupation",
    "https://www.theguardian.com/world/2026/jun/23/canada-montreal-shooting-gunman-three-killed-suspect-police-cote-des-neiges"
]

for i, url in enumerate(news_articles, 1):
    try:
        print(f"\nProcessing Article {i}...")
        
        article = Article(url)
        article.download()
        article.parse()
        
        print(f"Title: {article.title}")
        
        # 2. Convert text to tokens
        inputs = tokenizer(article.text, max_length=600, truncation=True, return_tensors="pt")

        # 3. Generate summary (Seq2Seq generation)
        if i == 1:
            # TEST 1 : short summary (max 20 words)
            print("Mode : Short summary")
            summary_ids = model.generate(
                inputs["input_ids"],
                max_new_tokens=60,      
                min_new_tokens=10,
                length_penalty=0.5,      
                num_beams=4,
                early_stopping=False,    
                max_length=None,         
                min_length=None
            )
        else:
            # TEST 2 : long (min 90 words)
            print("Mode : Long summary")
            summary_ids = model.generate(
                inputs["input_ids"],
                max_new_tokens=200,      
                min_new_tokens=90,
                length_penalty=2.0,
                num_beams=4,
                early_stopping=False
            )
        
        # 4. Convert generated tokens to text
        summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

        # Cutting the short summary at the last dot, because it tends to keep going on and stop in the middle of a sentence
        if i == 1:
            last_dot_index = summary_text.rfind(".")
            if last_dot_index != -1:
                summary_text = summary_text[:last_dot_index + 1]
        
        print(f"Summary:\n{summary_text}\n")
        print("-" * 60)
        
    except Exception as e:
        print(f"Error processing article {i}: {e}")

Loading model and tokenizer...


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]


Processing Article 1...
Title: Europe heatwave live: ‘London is cooking,’ says UN chief as UK forecast to hit 38C; France has hottest night since records began
Mode : Short summary


[transformers] Both `min_new_tokens` (=10) and `min_length`(=None) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summary:
UN secretary general, António Guterres, referenced Charles Dickens’ novel A Tale of Two Cities in a major address at London Climate Action Week.

------------------------------------------------------------

Processing Article 2...
Title: Ukraine intensifies attacks on Crimea to raise cost of Russian occupation
Mode : Long summary


[transformers] Both `max_new_tokens` (=200) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=90) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summary:
Ukrainian president Volodymyr Zelenskyy said a Crimean oil depot and an oil transport facility were among the targets. He described the attacks as part of Ukraine’s campaign of ‘long-range sanctions’ against Russia. Russia had already sharply restricted traffic on the Kerch Bridge, the other major route connecting Crimea to Russia. Moscow has largely stopped using the bridge for rail fuel shipments since a 2022 Ukrainian attack damaged the crossing and set a fuel train ablaze. The strikes have left residents queueing for hours at petrol stations, dealing a significant blow to Crimea's economy during the peak holiday season.

------------------------------------------------------------

Processing Article 3...
Title: ‘Nightmare’ shooting in Montreal leaves three dead including police officer and bystander
Mode : Long summary


[transformers] Both `max_new_tokens` (=200) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=90) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summary:
Police identified the deceased officer as constable Mohamed Lamine Benredouan, 34. He had been with the force since 2021. A second officer was seriously injured in the shooting in the city’s Côte-des-Neiges neighbourhood. The shooting occurred in a partly Jewish neighbourhood that includes kosher markets and restaurants, but police declined to comment on a possible motive. Canadian prime minister, Mark Carney, said he was “horrified’ by the violence.

------------------------------------------------------------
